# 02 — SHAP Analysis

Explain model predictions using SHAP values.

**Covers**: global importance (beeswarm + bar), local waterfall explanations, interaction dependence plots.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.extract import load_whas500
from src.data.transform import prepare_whas500_for_ml
from src.models.train import load_model
from src.models.shap_explainer import (
    build_explainer, compute_shap_values,
    global_feature_importance, local_explanation, interaction_analysis
)
from src.visualisation.plots import (
    plot_shap_summary, plot_shap_waterfall, plot_shap_dependence
)
from src.utils.config import WHAS500Config
print('Setup OK')

## 2. Load Model and Data

In [ ]:
whas_raw = load_whas500()
X_train, X_test, y_train, y_test, scaler = prepare_whas500_for_ml(whas_raw)

# Use XGBoost — best performing model
model = load_model('xgboost', 'whas500')
print(f'Model loaded. Test set: {len(X_test)} patients, {y_test.mean():.1%} mortality')

## 3. Build SHAP Explainer and Compute Values

In [ ]:
explainer   = build_explainer(model, X_train, 'xgboost')
shap_values = compute_shap_values(explainer, X_test, 'xgboost')
print(f'SHAP values shape: {shap_values.shape}')

## 4. Global Feature Importance

In [ ]:
importance = global_feature_importance(shap_values, list(X_test.columns))
print('Top 10 features by mean |SHAP|:')
for feat in importance:
    bar = '█' * int(feat['mean_abs_shap'] * 300)
    print(f"  {feat['rank']:2d}. {feat['feature']:<20} {feat['mean_abs_shap']:.4f}  {bar}")

In [ ]:
# SHAP beeswarm plot — the standard global importance visualisation
plot_shap_summary(shap_values, X_test, list(X_test.columns),
                  title='SHAP Summary — XGBoost (WHAS500 Mortality Prediction)',
                  plot_type='beeswarm')
plt.show()

## 5. Local Patient Explanation

In [ ]:
# Find the highest-risk patient in the test set
high_risk_idx = int(model.predict_proba(X_test)[:, 1].argmax())
expl = local_explanation(explainer, X_test, list(X_test.columns), patient_idx=high_risk_idx)

print(f'Patient {high_risk_idx}: predicted mortality risk = {expl["prediction"]:.1%}')
print(f'Baseline (mean prediction): {expl["base_value"]:.1%}')
print()
print('Top risk factors:')
for c in expl['top_risk_factors']:
    print(f"  {c['feature']:<20} value={c['value']:.2f}  SHAP=+{c['shap_value']:.4f}")
print()
print('Protective factors:')
for c in expl['top_protective']:
    print(f"  {c['feature']:<20} value={c['value']:.2f}  SHAP={c['shap_value']:.4f}")

## 6. Interaction: Age × Heart Rate

In [ ]:
plot_shap_dependence(
    shap_values, X_test,
    feature='age',
    interaction_feature='hr',
    title='SHAP Dependence: Age (coloured by Heart Rate)'
)
plt.show()
print('Interpretation: how does the effect of age on mortality risk\nvary with the patient\'s initial heart rate?')